# Regression with Multi-country Panel Data

This notebook is part of the English edition of the Big Data Analysis course materials.

This notebook uses a multi-country WDI panel to demonstrate variable selection, interpolation, descriptive statistics, correlation analysis, and OLS regression.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "04_regression_panel"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

# Q1. How can we prepare a multi-country panel dataset for regression analysis?

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

# Choose a group of countries for a panel-data example.
df_UC = df[df["Country Name"].isin([
    "Kazakhstan",
    "Kyrgyz Republic",
    "Turkiye",
    "Jordan",
    "Lebanon",
    "Israel",
    "Saudi Arabia",
    "Oman",
    "United Arab Emirates",
    "Qatar",
    "Kuwait",
    "Bahrain",
    "Egypt, Arab Rep.",
])]

WDI_UC = df_UC.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

### Convert Year from string-like values to integers.
WDI_UC["Year"] = WDI_UC["Year"].astype(str).astype(int)

isna_data = WDI_UC.isna().sum().sort_values(ascending=True)

isna_data.to_csv(MODULE_OUTPUT_DIR / "isna_data_UC.csv", index=True)

# Q2: How to select variables for a research question and perform statistical descriptions, Pearson correlation coefficients, regressions, and other analyses?

In [ ]:
WDI_reg = WDI_UC[
    [
        "Country Name",
        "Year",
        "GDP per capita (current US$)",
        "Foreign direct investment, net inflows (% of GDP)",
        "Trade (% of GDP)",
        "Inflation, consumer prices (annual %)",
    ]
].query("Year < 2021 and Year > 2005")

def interpolate_group(group):
    if group.isna().any().any():
        group = group.interpolate(method="linear", limit_direction="both")
    return group

df_interpolated = (
    WDI_reg.groupby("Country Name", group_keys=False)
    .apply(interpolate_group)
    .reset_index(drop=True)
)
df_interpolated.to_csv(MODULE_OUTPUT_DIR / "WDI_reg_interpolated.csv", index=False)

# Q3: How to do simple processing of data?

In [ ]:
import numpy as np
pd.options.mode.chained_assignment = None

### Log-transform GDP per capita.
df_interpolated["lngdpc"] = np.log(df_interpolated["GDP per capita (current US$)"])

df_interpolated.describe()

## 3.1 Select the data you want to describe and use describe to get descriptive statistics. Use T to transpose the data.

In [ ]:
### After obtaining a descriptive statistics table, `T` is useful for transposing the output into a more readable layout.

df_reg = df_interpolated.rename(columns={
    "Foreign direct investment, net inflows (% of GDP)": "FDI",
    "Trade (% of GDP)": "trade",
    "Inflation, consumer prices (annual %)": "inflation",
})

df_reg[["lngdpc", "FDI", "trade", "inflation"]].describe().round(2).T

## 3.2 The selected variables were analyzed for correlation. In general, variables with coefficients higher than 0.8 and significant are not selected for participation in the regression. 

### 3.2.1 How to do a correlation analysis and export it as a table with asterisks.

In [ ]:
### In practice, the matrix of correlation coefficients with significance asterisks can be obtained by modifying only the first line below.
cor_data = df_reg[["lngdpc", "FDI", "trade", "inflation"]]

from scipy.stats import pearsonr
import numpy as np

rho = cor_data.corr()

pval = cor_data.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*rho.shape)
p = pval.applymap(lambda x: "".join(["*" for t in [0.05, 0.01, 0.001] if x <= t]))
correlation = rho.round(2).astype(str) + p

correlation.to_csv(MODULE_OUTPUT_DIR / "correlation.csv", index=True)

correlation

### If you want to export to word format, please refer to the following link:
https://rowannicholls.github.io/python/data/export_to_word.html

### 3.2.2 The matrix of correlation coefficients can also be made into a heat map.

In [ ]:
%pip install -q seaborn

In [ ]:
import seaborn as sb
import matplotlib.pyplot as plt

dataplot = sb.heatmap(rho, cmap="Greens", annot=True)

plt.savefig(MODULE_OUTPUT_DIR / "heatmap.png", bbox_inches="tight")

# Q4: How to perform a simple OLS (ordinary least squares) regression?
https://www.statology.org/ols-regression-python/

In [ ]:
%pip install -q statsmodels

## 4.2 How to export regression results with a significant asterisk.

In [ ]:
%pip install -q openpyxl

In [ ]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

# Define the dependent variable.
y = df_reg[["lngdpc"]]

# Define the independent variables.
X1 = df_reg[["FDI"]]
X2 = df_reg[["FDI", "trade"]]
X3 = df_reg[["FDI", "trade", "inflation"]]

# Add constant terms.
X1 = sm.add_constant(X1)
X2 = sm.add_constant(X2)
X3 = sm.add_constant(X3)

# Fit the OLS models.
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()

# Create a summary table with significance stars.
reg = summary_col(
    [model1, model2, model3],
    stars=True,
    model_names=["Model 1", "Model 2", "Model 3"],
    float_format="%0.2f",
    info_dict={"N": lambda result: result.nobs},
)

reg_df = reg.tables[0].reset_index(drop=False)

# Export the summary table as an Excel file.
reg_df.to_excel(MODULE_OUTPUT_DIR / "regression_results.xlsx", index=False)

reg_df